In [2]:
import pandas as pd
import numpy as np
from pathlib import Path
import os

In [ ]:
results_dir = '../exp_results'

MODEL_CONFIGS = {
    # Main comparison models
    'moment': {
        'path': f'{results_dir}/vanilla_t5tiny/results.csv',
        'variants': {
            'u': {'mean_col': 'AutoMOMENT_vanilla_{metric}_mean', 'sd_col': 'AutoMOMENT_vanilla_{metric}_sd'},
        }
    },
    'patchtst': {
        'path': f'{results_dir}/vanilla_t5tiny/results.csv',
        'variants': {
            'u': {'mean_col': 'AutoPatchTSTMultivariate_vanilla_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_vanilla_{metric}_sd'},
        }
    },
    
    # Multivariate variants - HeadMixer
    'moment_hm': {
        'path': f'{results_dir}/vanilla_t5tiny/results.csv',
        'variants': {
            'hm': {'mean_col': 'AutoMOMENT_vanilla_headmixer_{metric}_mean', 'sd_col': 'AutoMOMENT_vanilla_headmixer_{metric}_sd'},
        }
    },
    'patchtst_hm': {
        'path': f'{results_dir}/vanilla_t5tiny/results.csv',
        'variants': {
            'hm': {'mean_col': 'AutoPatchTSTMultivariate_vanilla_headmixer_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_vanilla_headmixer_{metric}_sd'},
        }
    },
    
    # Adaptor - PCA
    'moment_pca': {
        'path': f'{results_dir}/vanilla_pca_t5tiny/results.csv',
        'variants': {
            'pca': {'mean_col': 'AutoMOMENT_vanilla_pca_{metric}_mean', 'sd_col': 'AutoMOMENT_vanilla_pca_{metric}_sd'},
        }
    },
    'patchtst_pca': {
        'path': f'{results_dir}/vanilla_pca_t5tiny/results.csv',
        'variants': {
            'pca': {'mean_col': 'AutoPatchTSTMultivariate_vanilla_pca_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_vanilla_pca_{metric}_sd'},
        }
    },

    # Baselines
    'itransformer': {
        'path': f'{results_dir}/itransformer_baseline/results.csv',
        'variants': {
            'm': {'mean_col': 'AutoiTransformer_multivariate_{metric}_mean', 'sd_col': 'AutoiTransformer_multivariate_{metric}_sd'},
        }
    },
    'itransformer_t5': {
        'path': f'{results_dir}/itransformer_baseline/results.csv',
        'variants': {
            'm': {'mean_col': 'AutoiTransformerT5_multivariate_{metric}_mean', 'sd_col': 'AutoiTransformerT5_multivariate_{metric}_sd'},
        }
    },
    'crossformer': {
        'path': f'{results_dir}/crossformer_baseline/results.csv',
        'variants': {
            'm': {'mean_col': 'AutoCrossformer_multivariate_{metric}_mean', 'sd_col': 'AutoCrossformer_multivariate_{metric}_sd'},
        }
    },
    'timerxl': {
        'path': f'{results_dir}/timerxl_baseline/results.csv',
        'variants': {
            'm': {'mean_col': 'AutoTimerXL_multivariate_{metric}_mean', 'sd_col': 'AutoTimerXL_multivariate_{metric}_sd'},
        }
    },
    'tsmixer': {
        'path': f'{results_dir}/tsmixer_baseline/results.csv',
        'variants': {
            'm': {'mean_col': 'AutoTSMixer_multivariate_{metric}_mean', 'sd_col': 'AutoTSMixer_multivariate_{metric}_sd'},
        }
    },
    'mlp': {
        'path': f'{results_dir}/multivariateMLP_baseline/results.csv',
        'variants': {
            'm': {'mean_col': 'AutoMLPMultivariate_multivariate_{metric}_mean', 'sd_col': 'AutoMLPMultivariate_multivariate_{metric}_sd'},
        }
    },
    'autoets': {
        'path': f'{results_dir}/statsforecast/results.csv',
        'variants': {
            'default': {'mean_col': 'AutoETS_{metric}_mean', 'sd_col': 'AutoETS_{metric}_sd'},
        }
    },
    'chronos2': {
        'path': f'{results_dir}/chronos2.0_baseline/results.csv',
        'variants': {
            'm': {'mean_col': 'Chronos_multivariate_{metric}_mean', 'sd_col': 'Chronos_multivariate_{metric}_sd'},
        }
    },
    
    # Infini ablation models
    'moment_shared_beta': {
        'path': f'{results_dir}/infini_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoMOMENT_infini_ciincl_{metric}_mean', 'sd_col': 'AutoMOMENT_infini_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoMOMENT_infini_ciexcl_{metric}_mean', 'sd_col': 'AutoMOMENT_infini_ciexcl_{metric}_sd'},
        }
    },
    'patchtst_shared_beta': {
        'path': f'{results_dir}/infini_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoPatchTSTMultivariate_infini_ciincl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_infini_ciincl_{metric}_sd'}, #Typo in name
            'excl': {'mean_col': 'AutoPatchTSTMultivariate_infini_ciexcl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_infini_ciexcl_{metric}_sd'}, #Typo in name
        }
    },

    'moment_channelwise_beta': {
        'path': f'{results_dir}/infini_channelwise_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoMOMENT_infini_channelwise_ciincl_{metric}_mean', 'sd_col': 'AutoMOMENT_infini_channelwise_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoMOMENT_infini_channelwise_ciexcl_{metric}_mean', 'sd_col': 'AutoMOMENT_infini_channelwise_ciexcl_{metric}_sd'},
        }
    },
    'patchtst_channelwise_beta': {
        'path': f'{results_dir}/infini_channelwise_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoPatchTSTMultivariate_infini_channelwise_ciincl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_infini_channelwise_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoPatchTSTMultivariate_infini_channelwise_ciexcl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_infini_channelwise_ciexcl_{metric}_sd'},
        }
    },

    'moment_layerwise_beta': {
        'path': f'{results_dir}/infini_layerwise_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoMOMENT_infini_layerwise_ciincl_{metric}_mean', 'sd_col': 'AutoMOMENT_infini_layerwise_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoMOMENT_infini_layerwise_ciexcl_{metric}_mean', 'sd_col': 'AutoMOMENT_infini_layerwise_ciexcl_{metric}_sd'},
        }
    },
    'patchtst_layerwise_beta': {
        'path': f'{results_dir}/infini_layerwise_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoPatchTSTMultivariate_infini_layerwise_ciincl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_infini_layerwise_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoPatchTSTMultivariate_infini_layerwise_ciexcl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_infini_layerwise_ciexcl_{metric}_sd'},
        }
    },

    'moment_layerwise_channelwise_beta': {
        'path': f'{results_dir}/infini_layerwise_channelwise_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoMOMENT_infini_layerwise_channelwise_ciincl_{metric}_mean', 'sd_col': 'AutoMOMENT_infini_layerwise_channelwise_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoMOMENT_infini_layerwise_channelwise_ciexcl_{metric}_mean', 'sd_col': 'AutoMOMENT_infini_layerwise_channelwise_ciexcl_{metric}_sd'},
        }
    },
    'patchtst_layerwise_channelwise_beta': {
        'path': f'{results_dir}/infini_layerwise_channelwise_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoPatchTSTMultivariate_infini_layerwise_channelwise_ciincl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_infini_layerwise_channelwise_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoPatchTSTMultivariate_infini_layerwise_channelwise_ciexcl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_infini_layerwise_channelwise_ciexcl_{metric}_sd'},
        }
    },

    'moment_mlp': {
        'path': f'{results_dir}/infini_mlpmixer_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoMOMENT_mlpmixer_ciincl_{metric}_mean', 'sd_col': 'AutoMOMENT_mlpmixer_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoMOMENT_mlpmixer_ciexcl_{metric}_mean', 'sd_col': 'AutoMOMENT_mlpmixer_ciexcl_{metric}_sd'},
        }
    },
    'patchtst_mlp': {
        'path': f'{results_dir}/infini_mlpmixer_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoPatchTSTMultivariate_mlpmixer_ciincl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_mlpmixer_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoPatchTSTMultivariate_mlpmixer_ciexcl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_mlpmixer_ciexcl_{metric}_sd'},
        }
    },

    'moment_mlpquery': {
        'path': f'{results_dir}/infini_mlpquerymixer_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoMOMENT_mlpquerymixer_ciincl_{metric}_mean', 'sd_col': 'AutoMOMENT_mlpquerymixer_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoMOMENT_mlpquerymixer_ciexcl_{metric}_mean', 'sd_col': 'AutoMOMENT_mlpquerymixer_ciexcl_{metric}_sd'},
        }
    },
    'patchtst_mlpquery': {
        'path': f'{results_dir}/infini_mlpquerymixer_t5tiny/results.csv',
        'variants': {
            'incl': {'mean_col': 'AutoPatchTSTMultivariate_mlpquerymixer_ciincl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_mlpquerymixer_ciincl_{metric}_sd'},
            'excl': {'mean_col': 'AutoPatchTSTMultivariate_mlpquerymixer_ciexcl_{metric}_mean', 'sd_col': 'AutoPatchTSTMultivariate_mlpquerymixer_ciexcl_{metric}_sd'},
        }
    },
}


TABLE_CONFIGS = {
    'main': {
        'models': [
            ('moment', ['u']),
            ('moment_mlpquery', ['incl']),
            ('patchtst', ['u']),
            ('patchtst_mlpquery', ['incl']),
            ('itransformer', ['m']),
            ('itransformer_t5', ['m']),
            ('crossformer', ['m']),
            ('timerxl', ['m']),
            ('tsmixer', ['m']),
            ('mlp', ['m']),
            ('chronos2', ['m']),
            ('autoets', ['default']),
        ],
    },
    'moment_infini_ablation': {
        'models': [
            ('moment_shared_beta', ['incl', 'excl']),
            ('moment_channelwise_beta', ['incl', 'excl']),
            ('moment_layerwise_beta', ['incl', 'excl']),
            ('moment_layerwise_channelwise_beta', ['incl', 'excl']),
            ('moment_mlp', ['incl', 'excl']),
            ('moment_mlpquery', ['incl', 'excl']),
        ],
    },
    'patchtst_infini_ablation': {
        'models': [
            ('patchtst_shared_beta', ['incl', 'excl']),
            ('patchtst_channelwise_beta', ['incl', 'excl']),
            ('patchtst_layerwise_beta', ['incl', 'excl']),
            ('patchtst_layerwise_channelwise_beta', ['incl', 'excl']),
            ('patchtst_mlp', ['incl', 'excl']),
            ('patchtst_mlpquery', ['incl', 'excl']),
        ],
    },
    'pca_ablation': {
        'models': [
            ('moment', ['u']),
            ('moment_pca', ['pca']),
            ('moment_hm', ['hm']),
            ('moment_mlpquery', ['incl']),
            ('patchtst', ['u']),
            ('patchtst_pca', ['pca']),
            ('patchtst_hm', ['hm']),
            ('patchtst_mlpquery', ['incl']),
        ],
    },
}


DATASET_MAPPING = {
        'covid_deaths': ('COVID Deaths', 'D'),
        'simglucose': ('Simglucose', '5min'),
        'iowa_ihop_smex_windspeed': ('Iowa IHOP SMEX02', '5min'),
        'iowa_plows_windspeed': ('Iowa PLOWS', '5min'),
        'jena_weather_D': ('Jena Weather', 'D'),
        'jena_weather_H': ('Jena Weather', 'H'),
        'M_DENSE_D': ('M-DENSE', 'D'),
        'M_DENSE_H': ('M-DENSE', 'H'),
        'LOOP_SEATTLE_H': ('Loop-Seattle', 'H'),
        'LOOP_SEATTLE_D': ('Loop-Seattle', 'D'),
        'ett1_H': ('ETT1', 'H'),
        'ett1_D': ('ETT1', 'D'),
        'ett1_W': ('ETT1', 'W'),
        'ett2_H': ('ETT2', 'H'),
        'ett2_D': ('ETT2', 'D'),
        'ett2_W': ('ETT2', 'W'),
        'solar_H': ('Solar', 'H'),
        'solar_D': ('Solar', 'D'),
        'solar_W': ('Solar', 'W'),
        'electricity_H': ('Electricity', 'H'),
        'electricity_D': ('Electricity', 'D'),
        'electricity_W': ('Electricity', 'W'),
    }


# Column rename map for mae_main table
MAE_MAIN_RENAME = {
    'AutoMOMENT_vanilla_mae_mean': 'moment_u',
    'AutoMOMENT_vanilla_mae_sd': 'moment_u_se',
    'AutoMOMENT_mlpquerymixer_ciincl_mae_mean': 'moment_m',
    'AutoMOMENT_mlpquerymixer_ciincl_mae_sd': 'moment_m_se',
    'AutoPatchTSTMultivariate_vanilla_mae_mean': 'patchtst_u',
    'AutoPatchTSTMultivariate_vanilla_mae_sd': 'patchtst_u_se',
    'AutoPatchTSTMultivariate_mlpquerymixer_ciincl_mae_mean': 'patchtst_m',
    'AutoPatchTSTMultivariate_mlpquerymixer_ciincl_mae_sd': 'patchtst_m_se',
    'AutoiTransformer_univariate_mae_mean': 'itransformer_u',
    'AutoiTransformer_univariate_mae_sd': 'itransformer_u_se',
    'AutoiTransformer_multivariate_mae_mean': 'itransformer_m',
    'AutoiTransformer_multivariate_mae_sd': 'itransformer_m_se',
    'AutoiTransformerT5_univariate_mae_mean': 'itransformer_t5_u',
    'AutoiTransformerT5_univariate_mae_sd': 'itransformer_t5_u_se',
    'AutoiTransformerT5_multivariate_mae_mean': 'itransformer_t5_m',
    'AutoiTransformerT5_multivariate_mae_sd': 'itransformer_t5_m_se',
    'AutoCrossformer_univariate_mae_mean': 'crossformer_u',
    'AutoCrossformer_univariate_mae_sd': 'crossformer_u_se',
    'AutoCrossformer_multivariate_mae_mean': 'crossformer_m',
    'AutoCrossformer_multivariate_mae_sd': 'crossformer_m_se',
    'AutoTimerXL_univariate_mae_mean': 'timerxl_u',
    'AutoTimerXL_univariate_mae_sd': 'timerxl_u_se',
    'AutoTimerXL_multivariate_mae_mean': 'timerxl_m',
    'AutoTimerXL_multivariate_mae_sd': 'timerxl_m_se',
    'AutoTSMixer_univariate_mae_mean': 'tsmixer_u',
    'AutoTSMixer_univariate_mae_sd': 'tsmixer_u_se',
    'AutoTSMixer_multivariate_mae_mean': 'tsmixer_m',
    'AutoTSMixer_multivariate_mae_sd': 'tsmixer_m_se',
    'AutoMLPMultivariate_univariate_mae_mean': 'mlp_u',
    'AutoMLPMultivariate_univariate_mae_sd': 'mlp_u_se',
    'AutoMLPMultivariate_multivariate_mae_mean': 'mlp_m',
    'AutoMLPMultivariate_multivariate_mae_sd': 'mlp_m_se',
    'AutoETS_mae_mean': 'autoets',
    'AutoETS_mae_sd': 'autoets_se',
}

# For RMSE, just replace 'mae' with 'rmse' in keys
RMSE_MAIN_RENAME = {k.replace('_mae_', '_rmse_'): v for k, v in MAE_MAIN_RENAME.items()}

MOMENT_INFINI_ABLATION_RENAME = {
    'AutoMOMENT_infini_ciincl_mae_mean': 'shared_incl',
    'AutoMOMENT_infini_ciincl_mae_sd': 'shared_incl_se',
    'AutoMOMENT_infini_ciexcl_mae_mean': 'shared_excl',
    'AutoMOMENT_infini_ciexcl_mae_sd': 'shared_excl_se',
    'AutoMOMENT_infini_channelwise_ciincl_mae_mean': 'channelwise_incl',
    'AutoMOMENT_infini_channelwise_ciincl_mae_sd': 'channelwise_incl_se',
    'AutoMOMENT_infini_channelwise_ciexcl_mae_mean': 'channelwise_excl',
    'AutoMOMENT_infini_channelwise_ciexcl_mae_sd': 'channelwise_excl_se',
    'AutoMOMENT_infini_layerwise_ciincl_mae_mean': 'layerwise_incl',
    'AutoMOMENT_infini_layerwise_ciincl_mae_sd': 'layerwise_incl_se',
    'AutoMOMENT_infini_layerwise_ciexcl_mae_mean': 'layerwise_excl',
    'AutoMOMENT_infini_layerwise_ciexcl_mae_sd': 'layerwise_excl_se',
    'AutoMOMENT_infini_layerwise_channelwise_ciincl_mae_mean': 'layerwise_channelwise_incl',
    'AutoMOMENT_infini_layerwise_channelwise_ciincl_mae_sd': 'layerwise_channelwise_incl_se',
    'AutoMOMENT_infini_layerwise_channelwise_ciexcl_mae_mean': 'layerwise_channelwise_excl',
    'AutoMOMENT_infini_layerwise_channelwise_ciexcl_mae_sd': 'layerwise_channelwise_excl_se',
    'AutoMOMENT_mlpmixer_ciincl_mae_mean': 'mlp_incl',
    'AutoMOMENT_mlpmixer_ciincl_mae_sd': 'mlp_incl_se',
    'AutoMOMENT_mlpmixer_ciexcl_mae_mean': 'mlp_excl',
    'AutoMOMENT_mlpmixer_ciexcl_mae_sd': 'mlp_excl_se',
    'AutoMOMENT_mlpquerymixer_ciincl_mae_mean': 'mlpquery_incl',
    'AutoMOMENT_mlpquerymixer_ciincl_mae_sd': 'mlpquery_incl_se',
    'AutoMOMENT_mlpquerymixer_ciexcl_mae_mean': 'mlpquery_excl',
    'AutoMOMENT_mlpquerymixer_ciexcl_mae_sd': 'mlpquery_excl_se',
}

PATCHTST_INFINI_ABLATION_RENAME = {
    k.replace('AutoMOMENT', 'AutoPatchTSTMultivariate'): v 
    for k, v in MOMENT_INFINI_ABLATION_RENAME.items()
}

PCA_ABLATION_RENAME = {
    'AutoMOMENT_vanilla_mae_mean': 'moment_vanilla',
    'AutoMOMENT_vanilla_mae_sd': 'moment_vanilla_se',
    'AutoMOMENT_vanilla_pca_mae_mean': 'moment_pca',
    'AutoMOMENT_vanilla_pca_mae_sd': 'moment_pca_se',
    'AutoMOMENT_vanilla_headmixer_mae_mean': 'moment_multivariatehead',
    'AutoMOMENT_vanilla_headmixer_mae_sd': 'moment_multivariatehead_se',
    'AutoMOMENT_mlpquerymixer_ciincl_mae_mean': 'moment_mica',
    'AutoMOMENT_mlpquerymixer_ciincl_mae_sd': 'moment_mica_se',
    'AutoPatchTSTMultivariate_vanilla_mae_mean': 'patchtst_vanilla',
    'AutoPatchTSTMultivariate_vanilla_mae_sd': 'patchtst_vanilla_se',
    'AutoPatchTSTMultivariate_vanilla_pca_mae_mean': 'patchtst_pca',
    'AutoPatchTSTMultivariate_vanilla_pca_mae_sd': 'patchtst_pca_se',
    'AutoPatchTSTMultivariate_vanilla_headmixer_mae_mean': 'patchtst_multivariatehead',
    'AutoPatchTSTMultivariate_vanilla_headmixer_mae_sd': 'patchtst_multivariatehead_se',
    'AutoPatchTSTMultivariate_mlpquerymixer_ciincl_mae_mean': 'patchtst_mica',
    'AutoPatchTSTMultivariate_mlpquerymixer_ciincl_mae_sd': 'patchtst_mica_se',
}


In [ ]:
def generate_table(model_configs, table_config, metric='mae', dataset_mapping=None, 
                   column_rename_map=None, datasets_to_include=None):
    """Generate a table for the specified models and metric.
    
    Args:
        model_configs: Dictionary of model configurations
        table_config: Configuration specifying which models/variants to include
        metric: Metric to use ('mae' or 'rmse')
        dataset_mapping: Optional mapping of dataset names to (display_name, frequency)
        column_rename_map: Optional mapping to rename columns
        datasets_to_include: Optional list of dataset names to include. If None, includes all datasets.
                            Datasets will appear in the order specified in this list.
    """
    
    # Load and concatenate all model data
    dfs = []
    column_names = []
    
    for model_name, variants in table_config['models']:
        config = model_configs[model_name]
        df = pd.read_csv(config['path'])

        for variant in variants:
            variant_config = config['variants'][variant]

            # Substitute metric into column names
            mean_col = variant_config['mean_col'].replace('{metric}', metric)
            std_col = variant_config['sd_col'].replace('{metric}', metric)
            
            # Store original column names
            column_names.append((mean_col, std_col))
            
            # Extract and rename columns
            subset = df[['dataset', mean_col, std_col]].copy()
            
            # Set AutoETS sd columns to NaN
            if 'autoets' in model_name.lower():
                subset[std_col] = np.nan
            
            dfs.append(subset)
    
    # Merge all dataframes on dataset
    result_df = dfs[0]
    for df in dfs[1:]:
        result_df = result_df.merge(df, on='dataset', how='outer')

    # Filter to specified datasets if provided
    if datasets_to_include is not None:
        result_df = result_df[result_df['dataset'].isin(datasets_to_include)].copy()
        result_df['dataset'] = pd.Categorical(result_df['dataset'], 
                                               categories=datasets_to_include, 
                                               ordered=True)
        result_df = result_df.sort_values('dataset').reset_index(drop=True)
    
    # Convert numeric columns to object dtype BEFORE any assignment
    mean_cols = [col for col, _ in column_names]
    std_cols = [col for _, col in column_names]
    
    for col in mean_cols + std_cols:
        result_df[col] = result_df[col].astype(object)
    
    # Find best and second best for each row
    for idx, row in result_df.iterrows():
        row_means = [row[col] for col in mean_cols]
        valid_means = [m for m in row_means if not pd.isna(m)]
        
        if len(valid_means) >= 2:
            sorted_means = sorted(valid_means)
            best_val = sorted_means[0]
            second_val = sorted_means[1]
        elif len(valid_means) == 1:
            best_val = valid_means[0]
            second_val = None
        else:
            continue
        
        # Mark best and second best with LaTeX formatting
        for col in mean_cols:
            if pd.isna(row[col]):
                result_df.at[idx, col] = '-'
            elif row[col] == best_val:
                result_df.at[idx, col] = f"\\textbf{{{row[col]:.3f}}}"
            elif row[col] == second_val:
                result_df.at[idx, col] = f"\\underline{{{row[col]:.3f}}}"
            else:
                result_df.at[idx, col] = f"{row[col]:.3f}"
        
        # Format std columns
        for std_col in std_cols:
            if pd.isna(row[std_col]):
                result_df.at[idx, std_col] = '-'
            else:
                result_df.at[idx, std_col] = f"{row[std_col]:.3f}"
    
    # Map dataset names and add frequency column if mapping provided
    if dataset_mapping is not None:
        result_df['Dataset'] = result_df['dataset'].map(lambda x: dataset_mapping.get(x, (x, '-'))[0])
        result_df['Frequency'] = result_df['dataset'].map(lambda x: dataset_mapping.get(x, (x, '-'))[1])
        
        # Merge duplicate dataset names - only show dataset name on first occurrence
        prev_dataset = None
        for idx in range(len(result_df)):
            current_dataset = result_df.at[idx, 'Dataset']
            if current_dataset == prev_dataset:
                result_df.at[idx, 'Dataset'] = ''
            else:
                prev_dataset = current_dataset
        
        # Drop original dataset column and reorder
        result_df = result_df.drop(columns=['dataset'])
        cols = ['Dataset', 'Frequency'] + [col for col in result_df.columns if col not in ['Dataset', 'Frequency']]
        result_df = result_df[cols]
    
    # Rename columns if mapping provided
    if column_rename_map is not None:
        result_df = result_df.rename(columns=column_rename_map)
    
    return result_df

In [31]:
def apply_improvement_coloring(result_df, metric='mae', comparison_pairs=None):
    """Apply blue coloring to cells where infini variants outperform vanilla variants.
    
    Args:
        result_df: DataFrame with formatted table (already has best/second-best formatting)
        metric: Metric being used ('mae' or 'rmse')
        comparison_pairs: Optional list of tuples (vanilla_col, infini_col) to compare.
                         If None, uses default MOMENT and PatchTST comparisons.
    
    Returns:
        DataFrame with blue coloring applied to improved infini variants
    """
    
    # Default comparison pairs if none provided
    if comparison_pairs is None:
        comparison_pairs = [
            (f'moment_u', 
             f'moment_m'),
            (f'patchtst_u', 
             f'patchtst_m')
        ]
    
    # Create a copy to avoid modifying the original
    df = result_df.copy()
    
    # Store original numeric values before they were formatted
    # We'll need to extract them from the formatted strings
    for vanilla_col, infini_col in comparison_pairs:
        # Check if both columns exist
        if vanilla_col not in df.columns or infini_col not in df.columns:
            continue
        
        for idx in df.index:
            vanilla_val = df.at[idx, vanilla_col]
            infini_val = df.at[idx, infini_col]
            
            # Skip if either value is missing or '-'
            if vanilla_val == '-' or infini_val == '-':
                continue
            
            # Extract numeric values from formatted strings
            vanilla_num = _extract_numeric_value(vanilla_val)
            infini_num = _extract_numeric_value(infini_val)
            
            if vanilla_num is None or infini_num is None:
                continue
            
            # If vanilla > infini (infini is better), color infini blue
            if vanilla_num > infini_num:
                df.at[idx, infini_col] = f"\\textcolor{{blue}}{{{infini_val}}}"
    
    return df


def _extract_numeric_value(formatted_str):
    """Extract numeric value from a formatted LaTeX string.
    
    Args:
        formatted_str: String that may contain LaTeX formatting
    
    Returns:
        Float value or None if extraction fails
    """
    import re
    
    if not isinstance(formatted_str, str):
        return formatted_str
    
    # Try to extract number from various LaTeX formats
    # Patterns: \textbf{0.123}, \underline{0.123}, or plain 0.123
    patterns = [
        r'\\textbf\{([\d.]+)\}',
        r'\\underline\{([\d.]+)\}',
        r'^([\d.]+)$'
    ]
    
    for pattern in patterns:
        match = re.search(pattern, formatted_str)
        if match:
            try:
                return float(match.group(1))
            except ValueError:
                continue
    
    return None

In [ ]:
datasets_for_main_table = [
    'simglucose',
    'covid_deaths',
    'iowa_ihop_smex_windspeed',
    'iowa_plows_windspeed',
    'jena_weather_H',
    'jena_weather_D',
    'M_DENSE_H',
    'M_DENSE_D',
    'LOOP_SEATTLE_D',
    'ett1_H',
    'ett1_D',
    'ett1_W',
    'ett2_H',
    'ett2_D',
    'ett2_W',
    'solar_H',
    'solar_D',
    'solar_W',
]

datasets_for_ablations = [
    'simglucose',
    'iowa_ihop_smex_windspeed',
    'iowa_plows_windspeed',
    'M_DENSE_H',
    'M_DENSE_D',
    'jena_weather_H',
    'jena_weather_D',
    'ett1_H',
    'ett1_D',
    'ett1_W',
    'ett2_H',
    'ett2_D',
    'ett2_W',
]

mae_main_table_df = generate_table(
    MODEL_CONFIGS,
    TABLE_CONFIGS['main'],
    metric='mae',
    dataset_mapping=DATASET_MAPPING,
    column_rename_map=MAE_MAIN_RENAME,
    datasets_to_include=datasets_for_main_table
)
mae_main_table_df = apply_improvement_coloring(mae_main_table_df, metric='mae')

rmse_main_table_df = generate_table(
    MODEL_CONFIGS,
    TABLE_CONFIGS['main'],
    metric='rmse',
    dataset_mapping=DATASET_MAPPING,
    column_rename_map=RMSE_MAIN_RENAME,
    datasets_to_include=datasets_for_main_table
)
rmse_main_table_df = apply_improvement_coloring(rmse_main_table_df, metric='rmse')

moment_infini_table_df = generate_table(
    MODEL_CONFIGS, 
    TABLE_CONFIGS['moment_infini_ablation'], 
    metric='mae', dataset_mapping=DATASET_MAPPING, 
    column_rename_map=MOMENT_INFINI_ABLATION_RENAME,
    datasets_to_include=datasets_for_ablations,
)

patchtst_infini_table_df = generate_table(
    MODEL_CONFIGS, 
    TABLE_CONFIGS['patchtst_infini_ablation'], 
    metric='mae', 
    dataset_mapping=DATASET_MAPPING, 
    column_rename_map=PATCHTST_INFINI_ABLATION_RENAME,
    datasets_to_include=datasets_for_ablations,
)

pca_ablation_table_df = generate_table(
    MODEL_CONFIGS, 
    TABLE_CONFIGS['pca_ablation'],
    metric='mae',
    dataset_mapping=DATASET_MAPPING, 
    column_rename_map=PCA_ABLATION_RENAME,
    datasets_to_include=datasets_for_ablations,
)


In [ ]:
os.makedirs(f'{results_dir}/paper_tables', exist_ok=True)

mae_main_table_df.to_csv(f'{results_dir}/paper_tables/mae_main.csv', index=False)
rmse_main_table_df.to_csv(f'{results_dir}/paper_tables/rmse_main.csv', index=False)
moment_infini_table_df.to_csv(f'{results_dir}/paper_tables/mae_moment_infini_ablation.csv', index=False)
patchtst_infini_table_df.to_csv(f'{results_dir}/paper_tables/mae_patchtst_infini_ablation.csv', index=False)
pca_ablation_table_df.to_csv(f'{results_dir}/paper_tables/pca_ablation.csv', index=False)

## Model Ranks

In [ ]:
results_dir = '../exp_results/' 

vanilla_results = pd.read_csv(f'{results_dir}/vanilla_t5tiny/results.csv')
mica_results = pd.read_csv(f'{results_dir}/infini_mlpquerymixer_t5tiny/results.csv')
itransformer_results = pd.read_csv(f'{results_dir}/itransformer_baseline/results.csv')
timerxl_results = pd.read_csv(f'{results_dir}/timerxl_baseline/results.csv')
crossformer_results = pd.read_csv(f'{results_dir}/crossformer_baseline/results.csv')
tsmixer_results = pd.read_csv(f'{results_dir}/tsmixer_baseline/results.csv')
mlp_results = pd.read_csv(f'{results_dir}/multivariateMLP_baseline/results.csv')
chronos_results = pd.read_csv(f'{results_dir}/chronos2.0_baseline/results.csv')

metric = 'rmse' #'mae'
table = pd.concat([
    vanilla_results[['dataset', f'AutoMOMENT_vanilla_{metric}_mean']].set_index('dataset'),
    vanilla_results[['dataset', f'AutoPatchTSTMultivariate_vanilla_{metric}_mean']].set_index('dataset'),
    mica_results[['dataset', f'AutoMOMENT_mlpquerymixer_ciincl_{metric}_mean']].set_index('dataset'),
    mica_results[['dataset', f'AutoPatchTSTMultivariate_mlpquerymixer_ciincl_{metric}_mean']].set_index('dataset'),
    itransformer_results[['dataset', f'AutoiTransformer_multivariate_{metric}_mean']].set_index('dataset'),
    itransformer_results[['dataset', f'AutoiTransformerT5_multivariate_{metric}_mean']].set_index('dataset'),
    timerxl_results[['dataset', f'AutoTimerXL_multivariate_{metric}_mean']].set_index('dataset'),
    crossformer_results[['dataset', f'AutoCrossformer_multivariate_{metric}_mean']].set_index('dataset'),
    tsmixer_results[['dataset', f'AutoTSMixer_multivariate_{metric}_mean']].set_index('dataset'),   
    mlp_results[['dataset', f'AutoMLPMultivariate_multivariate_{metric}_mean']].set_index('dataset'),
    chronos_results[['dataset', f'Chronos_multivariate_{metric}_mean']].set_index('dataset')
    ], axis=1
)

table = table.rank(axis=1, method='min')
table = table.mean(axis=0)
table = table.sort_values(ascending=True)
table.round(3)

In [ ]:
results_dir = '../exp_results/' 

vanilla_results = pd.read_csv(f'{results_dir}/vanilla_t5tiny/results.csv')
pca_results = pd.read_csv(f'{results_dir}/vanilla_pca_t5tiny/results.csv')
mica_results = pd.read_csv(f'{results_dir}/infini_mlpquerymixer_t5tiny/results.csv')

datasets = [
    'simglucose', 
    'iowa_ihop_smex_windspeed', 
    'iowa_plows_windspeed',
    'M_DENSE_H',
    'M_DENSE_D', 
    'jena_weather_H', 
    'jena_weather_D',
    'ett1_H', 
    'ett1_D', 
    'ett1_W', 
    'ett2_H', 
    'ett2_D', 
    'ett2_W',
]

metric = 'mae'
table = pd.concat([
    vanilla_results[['dataset', f'AutoMOMENT_vanilla_{metric}_mean']].set_index('dataset'),
    vanilla_results[['dataset', f'AutoPatchTSTMultivariate_vanilla_{metric}_mean']].set_index('dataset'),
    mica_results[['dataset', f'AutoMOMENT_mlpquerymixer_ciincl_{metric}_mean']].set_index('dataset'),
    mica_results[['dataset', f'AutoPatchTSTMultivariate_mlpquerymixer_ciincl_{metric}_mean']].set_index('dataset'),
    vanilla_results[['dataset', f'AutoMOMENT_vanilla_headmixer_{metric}_mean']].set_index('dataset'),
    vanilla_results[['dataset', f'AutoPatchTSTMultivariate_vanilla_headmixer_{metric}_mean']].set_index('dataset'),
    pca_results[['dataset', f'AutoMOMENT_vanilla_pca_{metric}_mean']].set_index('dataset'),
    pca_results[['dataset', f'AutoPatchTSTMultivariate_vanilla_pca_{metric}_mean']].set_index('dataset'),
    ], axis=1
)

table = table.loc[datasets]
table = table.rank(axis=1, method='min')
table = table.mean(axis=0)
table = table.sort_values(ascending=True)
table.round(3)